# 07. 위험 유형 클러스터링 2차

`07_risk_clustering_v2.py`를 셀 단위로 나눈 것. 로직은 원본과 동일하다.

1차 실패 교훈 반영:
1. 시나리오를 상한 clamp 아래 "중간 강도"로 (envType 가중 신호 보존)
2. 강수/강풍 분리 (수변 vs 해안·산악 신호 분리)
3. cat2 원핫 블록 가중 축소 (전체에서 컬럼 2개 분량으로) — 원핫 지배 방지
4. k 선택: silhouette + bootstrap 안정성(ARI) 병행

> ⚠️ 입력 피처가 점수 엔진의 출력이라 **가중치의 근거로 인용할 수 없다**(순환).
> 이 군집이 주장하는 것은 "규칙이 뭉뚱그린 구간을 표고·응급실 거리로 갈라냈다"까지다.
> 자세한 것은 `README.md`의 ⚠️ 절 참고.

**전제**: `places_features.parquet`가 있어야 한다. 없으면 `01_eda.py` → `03_build_codebook.py` 순으로 먼저 실행.

In [2]:
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

from safety_engine import compute_safety_score

# 노트북에는 __file__이 없다 — analysis/ 에서 실행하는 것을 전제로 cwd를 쓴다
HERE = Path.cwd()
SEED = 42

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 1. 데이터 로드 — 관광지(contentTypeId=12)만

In [3]:
df = pd.read_parquet(HERE / "places_features.parquet")
attr = df[df.contentTypeId == 12].reset_index(drop=True).copy()
print(f"대상: 관광지 {len(attr)}건")
attr[["contentId", "title", "envType", "elevation_m", "er_km", "cat2_name"]].head()

대상: 관광지 775건


,contentId,title,envType,elevation_m,er_km,cat2_name
0,2994116,가곡유황온천&스파,outdoor_general,108.0,18.657839,휴양관광지
1,2761729,가람리조트,outdoor_general,186.0,19.526290,휴양관광지
2,3068424,가리왕산케이블카,outdoor_mountain,425.0,11.023465,체험관광지
3,2714889,가원습지 생태자연공원,outdoor_general,25.0,6.254182,자연관광지
4,3041720,간이해변,outdoor_coast,0.0,1.465552,자연관광지


## 2. 중간 강도 시나리오

모든 값을 감점 상한 아래로 둔다. 극한 시나리오를 쓰면 clamp에 걸려 envType 가중 신호가 지워지기 때문 — 1차 실패의 원인이었다.

In [4]:
BASE = {"tempC": 24, "rainProbPct": 10, "windMs": 2, "pm25": 10, "forestFireLevel": 1}
SCEN = {
    "s_heat": ({"tempC": 34}, "heat"),                            # 주의보 구간 중간
    "s_rain": ({"rainProbPct": 60, "rainMm": 30}, "rain_wind"),   # 비만 — 수변 신호
    "s_wind": ({"windMs": 10}, "rain_wind"),                      # 바람만 — 해안·산악 신호
    "s_pm":   ({"pm25": 60}, "pm"),                               # 나쁨 등급
    "s_fire": ({"forestFireLevel": 3}, "fire"),                   # 3단계 — 산악 신호
}
for name, (over, key) in SCEN.items():
    attr[name] = [
        compute_safety_score({**BASE, **over, "emergencyRoomKm": km}, et)[key]
        for km, et in zip(attr.er_km, attr.envType)
    ]
attr["s_medical"] = [
    compute_safety_score({**BASE, "emergencyRoomKm": km}, et)["medical"]
    for km, et in zip(attr.er_km, attr.envType)
]

sens_cols = ["s_heat", "s_rain", "s_wind", "s_pm", "s_fire", "s_medical"]
print("시나리오 감점의 envType별 평균 (신호 보존 확인):")
attr.groupby("envType")[sens_cols].mean().round(1)

시나리오 감점의 envType별 평균 (신호 보존 확인):


,s_heat,s_rain,s_wind,s_pm,s_fire,s_medical
envType,,,,,,
indoor,5.0,3.0,1.0,2.0,12.0,1.8
outdoor_coast,17.0,11.0,6.0,8.0,12.0,3.5
outdoor_general,17.0,11.0,4.0,8.0,12.0,2.8
outdoor_mountain,17.0,11.0,5.0,8.0,16.0,3.5
outdoor_water,17.0,16.0,4.0,8.0,12.0,3.7


## 3. 피처 구성

민감도 6개 + 표고 1개를 표준화하고, cat2 원핫은 √(2/컬럼수)를 곱해 총 2컬럼 분량으로 축소한다. 그대로 두면 원핫이 거리 계산을 지배한다.

In [5]:
num = StandardScaler().fit_transform(attr[sens_cols + ["elevation_m"]])
cat2 = pd.get_dummies(attr.cat2_name, prefix="c2").astype(float)
cat2_s = StandardScaler().fit_transform(cat2) * np.sqrt(2 / cat2.shape[1])
X = np.hstack([num, cat2_s])
print(f"피처: 민감도 6 + 표고 1 + cat2 {cat2.shape[1]}(블록가중 √(2/{cat2.shape[1]})) = {X.shape[1]}차원")

피처: 민감도 6 + 표고 1 + cat2 7(블록가중 √(2/7)) = 14차원


## 4. k 선택 — silhouette + bootstrap ARI

**k=6의 근거가 되는 셀이다.**

k를 3부터 9까지 돌리며 두 지표를 본다.

- **silhouette** — 군집이 얼마나 조밀하고 서로 떨어져 있나 (응집도)
- **bootstrap ARI** — 데이터를 다시 뽑아도 같은 군집이 나오나 (안정성). 20회 반복

선택 규칙은 **ARI 0.6 이상인 것 중 silhouette 최대** — 안정성을 먼저 거른 뒤 응집도를 본다.

> ARI가 높다는 것은 "이 군집이 옳다"가 아니라 "재현된다"는 뜻이다. 안정성이지 타당성이 아니다.

(bootstrap 20회 × k 7개 = KMeans 140회, 수십 초 걸린다)

In [6]:
rng = np.random.default_rng(SEED)
print("k별 silhouette / bootstrap 안정성(ARI, 20회):")
metrics = {}
for k in range(3, 10):
    km = KMeans(n_clusters=k, n_init=20, random_state=SEED).fit(X)
    sil = silhouette_score(X, km.labels_)
    aris = []
    for b in range(20):
        idx = rng.choice(len(X), len(X), replace=True)
        kb = KMeans(n_clusters=k, n_init=10, random_state=b).fit(X[idx])
        # bootstrap 모델로 전체를 예측해 원 라벨과 비교
        aris.append(adjusted_rand_score(km.labels_, kb.predict(X)))
    metrics[k] = (sil, np.mean(aris))
    print(f"  k={k}: sil={sil:.3f}, ARI={np.mean(aris):.3f}")

# 선택 규칙: ARI 0.6 이상 중 silhouette 최대 (안정성 우선)
stable = {k: v for k, v in metrics.items() if v[1] >= 0.6} or metrics
best_k = max(stable, key=lambda k: stable[k][0])
print(f"→ 선택 k={best_k}")

k별 silhouette / bootstrap 안정성(ARI, 20회):
  k=3: sil=0.299, ARI=0.907
  k=4: sil=0.283, ARI=0.805
  k=5: sil=0.342, ARI=0.901
  k=6: sil=0.341, ARI=0.921
  k=7: sil=0.310, ARI=0.661
  k=8: sil=0.326, ARI=0.700
  k=9: sil=0.351, ARI=0.617
→ 선택 k=9


### k 선택 근거 표

위 출력을 표로 다시 본다. 원본 스크립트에는 없는 셀 — 계산은 하지 않고 `metrics`를 정리해 보여주기만 한다.

In [7]:
sel = pd.DataFrame(
    [{"k": k, "silhouette": round(v[0], 3), "bootstrap_ARI": round(v[1], 3)} for k, v in metrics.items()]
).set_index("k")
sel["ARI≥0.6"] = np.where(sel.bootstrap_ARI >= 0.6, "통과", "탈락")
sel["선택"] = np.where(sel.index == best_k, "◀ 채택", "")
sel

,silhouette,bootstrap_ARI,ARI≥0.6,선택
k,,,,
3,0.299,0.907,통과,
4,0.283,0.805,통과,
5,0.342,0.901,통과,
6,0.341,0.921,통과,
7,0.310,0.661,통과,
8,0.326,0.700,통과,
9,0.351,0.617,통과,◀ 채택


## 5. 최종 클러스터링

In [8]:
km = KMeans(n_clusters=best_k, n_init=50, random_state=SEED).fit(X)
attr["cluster"] = km.labels_
attr.cluster.value_counts().sort_index()

cluster
0    113
1    244
2     17
3    121
4    100
5     65
6     90
7     11
8     14
Name: count, dtype: int64

## 6. 군집 프로파일

각 군집이 어떤 위험에 민감한지, 표고와 응급실 거리는 어떤지. **고지·오지형**은 표고와 er_km가 함께 높은 군집이다.

In [9]:
prof = attr.groupby("cluster").agg(
    n=("contentId", "size"),
    heat=("s_heat", "mean"), rain=("s_rain", "mean"), wind=("s_wind", "mean"),
    pm=("s_pm", "mean"), fire=("s_fire", "mean"), medical=("s_medical", "mean"),
    elev=("elevation_m", "mean"), er_km=("er_km", "mean"),
).round(1)
prof

,n,heat,rain,wind,pm,fire,medical,elev,er_km
cluster,,,,,,,,,
0,113,17.0,11.0,4.0,8.0,12.0,6.2,473.3,22.1
1,244,17.0,11.0,4.0,8.0,12.0,1.5,176.4,7.3
2,17,5.0,3.0,1.0,2.0,12.0,1.8,261.2,8.7
3,121,17.0,11.0,5.0,8.0,16.0,3.5,408.5,13.9
4,100,17.0,11.0,6.0,8.0,12.0,3.5,19.2,13.4
5,65,17.0,11.0,4.0,8.0,12.0,1.9,131.5,8.1
6,90,17.0,16.0,4.0,8.0,12.0,3.7,286.7,14.3
7,11,17.0,11.0,4.1,8.0,12.4,1.7,399.2,8.5
8,14,17.0,11.4,4.1,8.0,12.3,3.5,284.1,13.1


### 군집 × envType 교차표

규칙(envType)이 `outdoor_general`로 뭉뚱그린 것이 어느 군집으로 흩어지는지 보는 표다. 규칙과 수렴한 군집(산악·해안·수변·실내)은 순환 때문에 필연이고, **의미는 general이 갈라진 쪽에 있다.**

In [10]:
pd.crosstab(attr.cluster, attr.envType)

envType,indoor,outdoor_coast,outdoor_general,outdoor_mountain,outdoor_water
cluster,,,,,
0,0,1,112,0,0
1,0,0,244,0,0
2,17,0,0,0,0
3,0,0,0,121,0
4,0,100,0,0,0
5,0,0,65,0,0
6,0,0,0,0,90
7,0,0,10,1,0
8,0,0,12,1,1


In [11]:
print("군집 × 시군 (상위 3):")
for c in sorted(attr.cluster.unique()):
    sub = attr[attr.cluster == c]
    top_sig = dict(sub.sigungu.value_counts().head(3))
    top_cat = dict(sub.cat3_name.value_counts().head(3))
    samples = sub.title.sample(min(4, len(sub)), random_state=SEED).tolist()
    print(f"\n[군집 {c}] {len(sub)}건 — 시군: {top_sig}")
    print(f"  소분류: {top_cat}")
    print(f"  예시: {samples}")

군집 × 시군 (상위 3):

[군집 0] 113건 — 시군: {'평창군': np.int64(40), '양양군': np.int64(13), '홍천군': np.int64(11)}
  소분류: {'농.산.어촌 체험': np.int64(34), '자연생태관광지': np.int64(12), '사찰': np.int64(10)}
  예시: ['켄싱턴 프렌치 가든', '고랭지만두마을', '샘재골송이마을', '인터컨티넨탈 알펜시아 평창 리조트']

[군집 1] 244건 — 시군: {'춘천시': np.int64(37), '원주시': np.int64(26), '강릉시': np.int64(25)}
  소분류: {'유적지/사적지': np.int64(54), '농.산.어촌 체험': np.int64(36), '이색체험': np.int64(29)}
  예시: ['근화동396청년창업공간', '강릉 해살이마을', '원주 흥양리 마애불좌상', '토성민속마을']

[군집 2] 17건 — 시군: {'평창군': np.int64(3), '원주시': np.int64(3), '강릉시': np.int64(2)}
  소분류: {'동굴': np.int64(9), '전통체험': np.int64(2), '안보관광': np.int64(1)}
  예시: ['강릉통일공원안보전시관', '고씨굴 (강원고생대 국가지질공원)', '삼척 초당굴', '화암동굴 (강원고생대 국가지질공원)']

[군집 3] 121건 — 시군: {'평창군': np.int64(18), '춘천시': np.int64(11), '영월군': np.int64(10)}
  소분류: {'산': np.int64(35), '자연휴양림': np.int64(17), '농.산.어촌 체험': np.int64(15)}
  예시: ['방태산', '봉래산(영월)', '건봉사', '산계리3층석탑']

[군집 4] 100건 — 시군: {'양양군': np.int64(26), '삼척시': np.int64(24), '고성군': np.int64(20)}
  소분류: {'해수욕장': np.

## 7. 저장

In [12]:
attr.to_parquet(HERE / "places_clustered_v2.parquet")
print("저장: analysis/places_clustered_v2.parquet")

저장: analysis/places_clustered_v2.parquet
